In [1]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from urllib.parse import urlparse
from tempfile import mkdtemp
from typing import Union
from enum import Enum
import numpy as np
import logging
import pickle
import shutil
import json
import os
import re

In [2]:
DATA_FILE = '/home/joao/my/ita/mestrado/2-clustering-phishing-kit/utils/data_filter_2.json'

In [29]:
from typing import List
from itertools import combinations

def cross_cosine_similarity(a: List[np.ndarray], b: List[np.ndarray]) -> float:
    max_similarity = -1
    most_similar_vectors = []

    for vector_a in a:
        for vector_b in b:
            similarity = cosine_similarity(vector_a.reshape(1, -1), vector_b.reshape(1, -1))
            if similarity > max_similarity:
                max_similarity = similarity
                most_similar_vectors = [(vector_a, vector_b)]

    return most_similar_vectors, max_similarity

def import_data(file):
    with open(file, 'r') as f:
        data = json.load(f)

    for filehash, segments in data.items():
        for idx, info in segments.items():
            info['vector'] = np.array(info['vector'], dtype=np.float32)

    hashes = [filehash for filehash in data.keys()]
    segmented_data = [[info['vector'] for info in segment.values()] for segment in data.values()]
    flat_data = np.array([info['vector'] for segment in data.values() for info in segment.values()], dtype=np.float32)

    # dm = cosine_similarity(flat_data, flat_data)
    # np.fill_diagonal(dm, 0.0)  # Replace diagonal values with 0.0

    choosen_hashes = []
    choosen_segments = []

    num_samples = len(segmented_data)
    total_segments_count = 0
    for i in range(num_samples):
        num_segments = len(segmented_data[i])
        current_segments = flat_data[total_segments_count:total_segments_count+num_segments]

        if num_segments == 0:
            continue
        
        # current_segments_distances = dm[total_segments_count:total_segments_count+num_segments, :]
        current_segments_distances = cosine_similarity(flat_data[total_segments_count:total_segments_count+num_segments], flat_data)
       
        diag_indices = np.diag_indices(num_segments)
        diag_indices = (diag_indices[0], diag_indices[1] + total_segments_count)
        current_segments_distances[diag_indices] = 0.0

        max_similarity_indices = np.unravel_index(np.argmax(current_segments_distances), current_segments_distances.shape)

        most_similar_vector_x = current_segments[max_similarity_indices[0]]

        choosen_hashes.append(hashes[i])
        choosen_segments.append(most_similar_vector_x)
        
        total_segments_count += num_segments

    # return segments
    return np.array(choosen_segments, dtype=np.float32), np.array(choosen_hashes)

In [30]:
X, y = import_data(DATA_FILE)

In [31]:
with open('vectors_partial.tsv', 'w') as f:
    for i in range(X.shape[0]):
        f.write('\t'.join([str(x) for x in X[i]]) + '\n')

In [32]:
from sklearn.cluster import DBSCAN

# Create an instance of DBSCAN
dbscan = DBSCAN(eps=0.05, min_samples=1, metric='cosine')

# Fit the data to DBSCAN
dbscan.fit(X)

# Get the labels assigned by DBSCAN
labels = dbscan.labels_

# Print the labels
print(labels)

[  0   1   2   3   4   5   6   7   8   9  10  11  12   9  13  14  15   4
   3  16   3   3  17  18   3   9  19   3  20  11  13  21   9  22  23  24
  25   3  26   1  27  28  29  14  30  31   9  32  33  34  35   9  36  37
  38   9  39  40  36  41  42   3  43  44  37  45  46  47  45  14  45   9
  37  37  37  37  36  48  49  50   9  36  51   4  36  14  48   9   4  52
  53  54  46  36  37  55  34  56  48  21  57  49   1  55  58  13  37  59
  37  36  36   9  60  57  61  62   7  55  63  37  48  64  13  14  58   4
  65   4   3  37  57  66   9  54  36  49  57   5   3   4  36  67  43  62
  68  69  36   4  70  71  72  73  74  75  76  77  49  78  79  80  37  43
  11   9  49  47  14  81  82  47  14  83  81  47  36   3  41  84  11  59
  18  54  44  77  85  41  86  50  87  88   9  89  22  90  91  92  63  93
  92  37  94  95  96  97  92   9  70  49  98  91  75  14  18  11  99  14
  49 100  97  67 101 102  47 103  36  37   3 104   9 105  26 106 107   9
  81 108 109 106]


In [33]:
with open('metadata_partial.tsv', 'w') as f:
    f.write('hash\tlabel\n')
    for i in range(y.shape[0]):
        f.write(y[i] + '\t' + str(labels[i]) + '\n')